You need to upload the ARD GSV images to a Google Cloud Storage Bucket to run this code

In [1]:
import base64
import json
import logging
import os
from pathlib import Path
from pprint import pformat

import pandas as pd

import vertexai
from google.cloud import storage
from vertexai.batch_prediction import BatchPredictionJob

In [2]:
def setup_logging(INTERMEDIATE_PATH, PHASE):
    """
    Set up logging configuration.
    
    Args:
        INTERMEDIATE_PATH (str): Directory to store log files.
        PHASE (str): Phase of analysis ("prompt_opt", "final_pred", or "experimental").
        
    Returns:
        Configured logger instance.
    """
    log_dir = INTERMEDIATE_PATH / PHASE
    log_dir.mkdir(parents=True, exist_ok=True)
    
    log_file = log_dir / "3_1_google.log"
    
    logging.basicConfig(
        level=logging.DEBUG,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

def get_response_schemas(GOOGLE_RESPONSE_SCHEMAS_PATH, PROMPTING_TYPE):
    """
    Load and process response schemas from a JSON file based on the prompting type.

    Parameters:
        GOOGLE_RESPONSE_SCHEMAS_PATH (str): Path to the JSON file containing response schemas.
        PROMPTING_TYPE (str): Type of prompting ('s_d', 'cot_d', 's_m', or 'cot_m').

    Returns:
        A dictionary containing relevant response schemas.
    """
    try:
        with open(GOOGLE_RESPONSE_SCHEMAS_PATH, "r") as file:
            schemas = json.load(file)
    except FileNotFoundError:
        logger.error(f"File not found: {GOOGLE_RESPONSE_SCHEMAS_PATH}")
        raise
    except json.JSONDecodeError:
        logger.error(f"Failed to decode JSON from file: {GOOGLE_RESPONSE_SCHEMAS_PATH}")
        raise

    logger.info(f"Loaded response schemas from {GOOGLE_RESPONSE_SCHEMAS_PATH}", )

    if PROMPTING_TYPE in ["s_d", "cot_d"]:
        response_schema_dict = {
            "response_schema_construction": schemas[PROMPTING_TYPE]["response_schema_construction"],
            "response_schema_currentuse": schemas[PROMPTING_TYPE]["response_schema_currentuse"],
            "response_schema_storeys": schemas[PROMPTING_TYPE]["response_schema_storeys"]
        }
        logger.debug(f"Response schema for {PROMPTING_TYPE} prompting:\n{pformat(response_schema_dict)}")
        logger.info("Selected response schema for prompting type: %s", PROMPTING_TYPE)

    elif PROMPTING_TYPE in ["s_m", "cot_m"]:
        response_schema_dict = {
            "response_schema_merged": schemas[PROMPTING_TYPE]["response_schema_merged"]
        }
        logger.debug(f"Response schema for {PROMPTING_TYPE} prompting:\n{pformat(response_schema_dict)}")
        logger.info("Selected merged response schema for prompting type: %s", PROMPTING_TYPE)

    else:
        logger.error(f"Unsupported prompting type: {PROMPTING_TYPE}")
        raise ValueError(f"Unsupported prompting type: {PROMPTING_TYPE}")

    return response_schema_dict

def typologies_prediction_helpers(TYPOLOGIES_PREDICTION_HELPER_PATH, PROMPTING_TYPE):
    """
    Load and process typologies prediction helpers from a JSON file based on the prompting type.

    Parameters:
        TYPOLOGIES_PREDICTION_HELPER_PATH (str): Path to the JSON file containing typologies prediction helpers.
        PROMPTING_TYPE (str): Type of prompting ('s_d', 'cot_d', 's_m', or 'cot_m').

    Returns:
        A tuple containing the token, prompt dictionary, system prompt dictionary, list of prompt keys, and dataset.
    """
    try:
        with open(TYPOLOGIES_PREDICTION_HELPER_PATH, "r") as file:
            data = json.load(file)[PROMPTING_TYPE]
    except FileNotFoundError:
        logger.error(f"File not found: {TYPOLOGIES_PREDICTION_HELPER_PATH}")
        raise
    except json.JSONDecodeError:
        logger.error(f"Failed to decode JSON from file: {TYPOLOGIES_PREDICTION_HELPER_PATH}")
        raise

    token = data["token"]
    logger.info(f"Token required with {PROMPTING_TYPE} prompting: {token}")

    if PROMPTING_TYPE in ["s_d", "cot_d"]:
        system_prompt_construction = base64.b64decode(data["system_prompt"]["construction"]).decode("utf-8")
        system_prompt_currentuse = base64.b64decode(data["system_prompt"]["currentuse"]).decode("utf-8")
        system_prompt_storeys = base64.b64decode(data["system_prompt"]["storeys"]).decode("utf-8")

        system_prompt_dict = {
            "construction": system_prompt_construction,
            "currentuse": system_prompt_currentuse,
            "storeys": system_prompt_storeys
        }

        logger.debug(f"Decoded system prompts for {PROMPTING_TYPE} prompting:\n{pformat(system_prompt_dict)}")

        prompt_construction = base64.b64decode(data["prompt"]["construction"]).decode("utf-8")
        prompt_currentuse = base64.b64decode(data["prompt"]["currentuse"]).decode("utf-8")
        prompt_storeys = base64.b64decode(data["prompt"]["storeys"]).decode("utf-8")

        prompt_dict = {
            "construction": prompt_construction,
            "currentuse": prompt_currentuse,
            "storeys": prompt_storeys
        }

        logger.debug(f"Decoded prompts for {PROMPTING_TYPE} prompting:\n{pformat(prompt_dict)}")

        prompt_dict_keys = list(prompt_dict.keys())

        dataset = pd.DataFrame({
            "id": [],
            "image_base64": [],
            "system_prompt_construction": [],
            "system_prompt_currentuse": [],
            "system_prompt_storeys": [],
            "prompt_construction": [],
            "prompt_currentuse": [],
            "prompt_storeys": []
        }).astype({
            "id": "int32",
            "image_base64": str,
            "system_prompt_construction": str,
            "system_prompt_currentuse": str,
            "system_prompt_storeys": str,
            "prompt_construction": str,
            "prompt_currentuse": str,
            "prompt_storeys": str
        })

    elif PROMPTING_TYPE in ["s_m", "cot_m"]:
        system_prompt_merged = base64.b64decode(data["system_prompt"]).decode("utf-8")

        system_prompt_dict = {
            "merged": system_prompt_merged
        }

        logger.debug(f"Decoded merged system prompt for {PROMPTING_TYPE} prompting:\n{pformat(system_prompt_dict)}")

        prompt_merged = base64.b64decode(data["prompt"]).decode("utf-8")

        prompt_dict = {
            "merged": prompt_merged
        }

        logger.debug(f"Decoded merged prompt for {PROMPTING_TYPE} prompting:\n{pformat(prompt_dict)}")

        prompt_dict_keys = list(prompt_dict.keys())

        dataset = pd.DataFrame({
            "id": [],
            "image_base64": [],
            "system_prompt_merged": [],
            "prompt_merged": []
        }).astype({
            "id": "int32",
            "image_base64": str,
            "system_prompt_merged": str,
            "prompt_merged": str
        })

    else:
        logger.error(f"Unsupported prompting type: {PROMPTING_TYPE}")
        raise ValueError(f"Unsupported prompting type: {PROMPTING_TYPE}")

    return token, prompt_dict, system_prompt_dict, prompt_dict_keys, dataset


def create_helper_df(ARD_GSV_PATH, PREDICT_COUNT, system_prompt_dict, prompt_dict, prompt_dict_keys, dataset):
    """
    Create a helper DataFrame by processing image files and updating the dataset with system prompts and prompts.

    Parameters:
        ARD_GSV_PATH (str): Path to the directory containing JPEG images.
        PREDICT_COUNT (int): Number of samples to predict.
        system_prompt_dict (dict): Dictionary containing system prompts.
        prompt_dict (dict): Dictionary containing prompts.
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).
        dataset (DataFrame): Initial helper dataset to be updated.

    Returns:
        Updated dataset DataFrame with image IDs and associated prompts.
    """
    jpeg_images = list(Path(ARD_GSV_PATH).iterdir())
    if not jpeg_images:
        logger.warning(f"No JPEG images found in {ARD_GSV_PATH}")
        return dataset

    img_ids = []
    for image in jpeg_images:
        if image.suffix == ".jpeg":
            img_id = int(image.name.split("_")[0])
            img_ids.append(img_id)
        else:
            logger.debug(f"Skipped non-JPEG file: {image.name}")

    dataset["id"] = img_ids
    logger.info(f"Loaded {len(img_ids)} image IDs from {ARD_GSV_PATH}")

    if not dataset["id"].is_unique:
        logger.error("Image IDs are not unique")
        raise ValueError("Duplicate image IDs found")

    if PROMPTING_TYPE in ["s_d", "cot_d"]:
        if len(prompt_dict_keys) != 3:
            logger.error("Prompt dictionary keys should be exactly three for 's_d' or 'cot_d'")
            raise ValueError("Invalid number of prompt dictionary keys")
        
        dataset["system_prompt_construction"] = system_prompt_dict[prompt_dict_keys[0]]
        dataset["system_prompt_currentuse"] = system_prompt_dict[prompt_dict_keys[1]]
        dataset["system_prompt_storeys"] = system_prompt_dict[prompt_dict_keys[2]]
        
        dataset["prompt_construction"] = prompt_dict[prompt_dict_keys[0]]
        dataset["prompt_currentuse"] = prompt_dict[prompt_dict_keys[1]]
        dataset["prompt_storeys"] = prompt_dict[prompt_dict_keys[2]]
        
        logger.info("Updated dataset with construction, current use, and storeys prompts")

    elif PROMPTING_TYPE in ["s_m", "cot_m"]:
        if len(prompt_dict_keys) != 1:
            logger.error("Prompt dictionary keys should be exactly one for 's_m' or 'cot_m'")
            raise ValueError("Invalid number of prompt dictionary keys")
        
        dataset["system_prompt_merged"] = system_prompt_dict[prompt_dict_keys[0]]
        
        dataset["prompt_merged"] = prompt_dict[prompt_dict_keys[0]]
        
        logger.info("Updated dataset with merged prompts")

    else:
        logger.error(f"Unsupported prompting type: {PROMPTING_TYPE}")
        raise ValueError(f"Unsupported prompting type: {PROMPTING_TYPE}")

    dataset.sort_values("id", inplace=True)
    dataset.reset_index(drop=True, inplace=True)

    if PREDICT_COUNT > len(dataset):
        logger.warning(f"PREDICT_COUNT ({PREDICT_COUNT}) is greater than the number of available images ({len(dataset)}). Using all available images.")
        PREDICT_COUNT = len(dataset)

    dataset = dataset.sample(PREDICT_COUNT, random_state=123)
    logger.info(f"Sampled {PREDICT_COUNT} rows from the dataset")

    return dataset

def get_task_partitions(BUCKET_NAME, NUM_PARTITION, response_schema_dict, prompt_dict_keys, dataset, token):
    """
    Partition tasks based on the number of partitions and generate task dictionaries.

    Parameters:
        BUCKET_NAME (str): Name of the Google Cloud Storage bucket.
        NUM_PARTITION (int): Number of partitions.
        response_schema_dict (dict): Dictionary containing relevant response schemas.
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).
        dataset (DataFrame): DataFrame with image IDs and associated prompts.
        token (str): Token required for prompting.

    Returns:
        A dictionary containing task partitions.
    """
    # Add custom_id column
    dataset['custom_id'] = 'task-' + dataset.id.astype(str)    
    logger.info("Added custom_id column to the dataset")

    # Generate messages for each prompt type
    for typology in prompt_dict_keys:
        dataset[f'messages_{typology}'] = dataset.apply(
            lambda row: [
                {
                    "role": "user",
                    "parts": [
                        {
                            "text": row[f'prompt_{typology}']
                        },
                        {
                            "file_data": {
                              "file_uri": f"gs://{BUCKET_NAME}/{row['id']}_GSV.jpeg",
                              "mime_type": "image/jpeg"
                            }
                        }
                    ]
                }
            ], axis=1
        )
        logger.debug(f"Generated messages for {typology} prompt type")
    
    for typology in prompt_dict_keys:
        dataset[f'task_{typology}'] = dataset.apply(
            lambda row: {
                "request": {
                    "contents": row[f'messages_{typology}'],
                    "systemInstruction": {
                        "role": "assistant",
                        "parts": [
                            {
                                "text": row[f'system_prompt_{typology}']
                            }
                        ]
                    },
                    "generationConfig": {
                        "maxOutputTokens": token + 65,  # 65 accounting estimated tokens required to output the JSON's keys, brackets, commas, etc.
                        "responseMimeType": "application/json",
                        "responseSchema": response_schema_dict[f"response_schema_{typology}"],
                        "seed": 123
                    },
                    "labels": {
                        "label": f"{row['custom_id']}"
                    }
                }
            }, axis=1
        )
        logger.debug(f"Generated tasks for {typology} prompt type")
    
    tasks_dict = {}

    # Partition tasks
    for typology in prompt_dict_keys:
        task_list = dataset[f"task_{typology}"].tolist()
        task_partition_len = len(task_list) // NUM_PARTITION
        for i in range(0, NUM_PARTITION):
            task_list_start = task_partition_len * i
            task_list_end = task_partition_len * (i + 1) if i != NUM_PARTITION - 1 else len(task_list)
            tasks_dict[f"{typology}_partition_{i + 1}"] = task_list[task_list_start:task_list_end]
        logger.debug(f"Partitioned tasks for {typology} prompt type into {NUM_PARTITION} partitions")
    
    return tasks_dict

def upload_requests_to_gcs(BUCKET_NAME, INTERMEDIATE_DIR, NUM_PARTITION, PROMPTING_TYPE, PHASE, prompt_dict_keys):
    """
    Upload request files to Google Cloud Storage.

    Parameters:
        BUCKET_NAME (str): The name of the GCS bucket.
        INTERMEDIATE_DIR (str): Directory containing intermediate files.
        NUM_PARTITION (int): Number of partitions.
        PROMPTING_TYPE (str): Type of prompting.
        PHASE (str): Phase of analysis ("prompt_opt", "final_pred", or "experimental").
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).

    Returns:
        Requests stored to GCS.
    """
    # Initialize the Google Cloud Storage client
    try:
        storage_client = storage.Client()
    except Exception as e:
        logger.error(f"Failed to initialize Google Cloud Storage client: {e}")
        raise

    for typology in prompt_dict_keys:
        for i in range(0, NUM_PARTITION):
            file_name = INTERMEDIATE_DIR / PHASE / f"g_{PROMPTING_TYPE}_{typology}_partition_{i}.jsonl"
            
            with open(file_name, 'w') as file:
                for obj in tasks_dict[f"{typology}_partition_{i}"]:
                    file.write(json.dumps(obj) + '\n')
    
            # Name of the object in the bucket
            object_name = f"g_{PROMPTING_TYPE}_{typology}_partition_{i}.jsonl"

            # Check if the file exists
            if not file_name.exists():
                logger.warning(f"File does not exist: {file_name}")
                continue
            
            # Get the bucket instance
            try:
                bucket = storage_client.bucket(BUCKET_NAME)
            except Exception as e:
                logger.error(f"Failed to get bucket {BUCKET_NAME}: {e}")
                raise
            
            # Get the blob instance
            try:
                blob = bucket.blob(object_name)
            except Exception as e:
                logger.error(f"Failed to get blob {object_name}: {e}")
                raise

            # Upload the file
            try:
                blob.upload_from_filename(file_name)
                logger.info(f"File {file_name} uploaded to {BUCKET_NAME} as {object_name}")
            except Exception as e:
                logger.error(f"Failed to upload file {file_name} to {BUCKET_NAME}: {e}")
                raise

def submit_prediction_requests(BUCKET_URI, MODEL_ID, NUM_PARTITION, PROMPTING_TYPE, prompt_dict_keys):
    """
    Submit prediction requests to a model.

    Parameters:
        BUCKET_URI (str): The URI of the Google Cloud Storage bucket.
        MODEL_ID (str): ID of the model to use for predictions.
        NUM_PARTITION (int): Number of partitions.
        PROMPTING_TYPE (str): Type of prompting.
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).

    Returns:
        Requests submitted as batch predictions to Vertex AI.
    """
    for typology in prompt_dict_keys:
        for i in range(NUM_PARTITION):
            input_data = BUCKET_URI + f"/g_{PROMPTING_TYPE}_{typology}_partition_{i}.jsonl"
            input_path = Path(input_data)
            
            if not input_path.exists():
                logger.warning(f"File {input_data} does not exist. Skipping submission.")
                continue

            try:
                job = BatchPredictionJob.submit(
                    source_model=MODEL_ID,
                    input_dataset=input_data,
                    output_uri_prefix=BUCKET_URI
                )
                logger.info(f"Job submitted for {input_data} with model {MODEL_ID}.")
            except Exception as e:
                logger.error(f"Failed to submit job for {input_data}: {e}")

def download_prediction_results(BUCKET_NAME, NUM_PARTITION, PROMPTING_TYPE, job_dict, prompt_dict_keys):
    """
    Download prediction results from a Google Cloud Storage bucket.

    Parameters:
        BUCKET_NAME (str): The name of the GCS bucket.
        NUM_PARTITION (int): Number of partitions.
        PROMPTING_TYPE (str): Type of prompting.
        job_dict (dict): Dictionary containing job requests.
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).

    Returns:
        Prediction results saved.
    """
    # Initialize the Google Cloud Storage client
    try:
        storage_client = storage.Client()
    except Exception as e:
        logger.error(f"Failed to initialize Google Cloud Storage client: {e}")
        raise

    # Get the bucket instance
    try:
        bucket = storage_client.bucket(BUCKET_NAME)
    except Exception as e:
        logger.error(f"Failed to get bucket {BUCKET_NAME}: {e}")
        raise

    for typology in prompt_dict_keys:
        for i in range(NUM_PARTITION):            
            # Get the output info
            output_info = job_dict[f"{PROMPTING_TYPE}_{typology}_partition_{i}"]
            prefix = output_info.split('gs://')[-1].lstrip('/').split("/")[1]
            
            # List the blobs in the bucket
            blobs = list(bucket.list_blobs(prefix=prefix))
            
            # Download each blob to a local file
            download_path = INTERMEDIATE_DIR / PHASE / f"result_g_{PROMPTING_TYPE}_{typology}_partition_{i}.jsonl"
            for i, blob in enumerate(blobs):
                if not "chunked" in blob.name:
                    try:
                        blob.download_to_filename(download_path)
                        logger.info(f"Downloaded {blob.name} to {download_path}")
                    except Exception as e:
                        logger.error(f"Failed to download {blob.name}: {e}")
    
    logger.info("All prediction results have been downloaded.")

def parse_result_jsonl(INTERMEDIATE_DIR, PROMPTING_TYPE, PHASE, typology):
    """
    Parses JSONL files containing prediction results and combines them into a DataFrame.

    Args:
        INTERMEDIATE_DIR (Path): Directory containing the JSONL files.
        PROMPTING_TYPE (str): Type of prompting.
        PHASE (str): Phase of analysis ("prompt_opt", "final_pred", or "experimental").
        typology (str): Building typology predicted.

    Returns:
        A DataFrame containing parsed prediction results.
    """
    prediction_dict_list = []
    prediction_df = pd.DataFrame()

    for i in range(NUM_PARTITION):
        file_path = INTERMEDIATE_DIR / PHASE / f"result_g_{PROMPTING_TYPE}_{typology}_partition_{i}.jsonl"
        try:
            jsonObj = pd.read_json(path_or_buf=file_path, lines=True)
        except Exception as e:
            logging.error(f"Failed to read JSONL file {file_path}: {e}")
            continue

        for j in range(jsonObj.shape[0]):
            prediction_dict = {}
            try:
                prediction_dict["id"] = int(jsonObj.iloc[j, :].request["contents"][0]["parts"][1]["file_data"]["file_uri"].split("/")[3].split("_")[0])
            except (KeyError, IndexError) as e:
                logging.error(f"Failed to extract 'id' from line {j} in partition {i}: {e}")
                continue

            try:
                response_text = json.loads(jsonObj.iloc[j, :].response["candidates"][0]["content"]["parts"][0]["text"])
            except (KeyError, json.JSONDecodeError) as e:
                logging.error(f"Failed to decode JSON response in line {j} in partition {i}: {e}")
                continue

            if PROMPTING_TYPE == "s_d":
                prediction_dict[f"{typology}"] = response_text.get(f"{typology}", None)
                prediction_dict[f"{typology}_reason"] = response_text.get("reason", None)
            elif PROMPTING_TYPE == "cot_d":
                prediction_dict[f"{typology}_thoughts"] = response_text.get("thoughts", None)
                prediction_dict[f"{typology}"] = response_text.get(f"{typology}", None)
                prediction_dict[f"{typology}_reason"] = response_text.get("reason", None)
            elif PROMPTING_TYPE == "s_m":
                prediction_dict[f"{typology}"] = response_text.get(f"{typology}", None)
                prediction_dict[f"{typology}_reason"] = response_text.get(f"reason_{typology}", None)
            elif PROMPTING_TYPE == "cot_m":
                prediction_dict[f"{typology}_thoughts"] = response_text.get(f"thoughts_{typology}", None)
                prediction_dict[f"{typology}"] = response_text.get(f"{typology}", None)
                prediction_dict[f"{typology}_reason"] = response_text.get(f"reason_{typology}", None)

            prediction_dict_list.append(prediction_dict)

        helper_df = pd.DataFrame(prediction_dict_list, index=[i for i in range(0, len(prediction_dict_list))])
        prediction_df = pd.concat([prediction_df, helper_df], ignore_index=True)
        prediction_dict_list = []  # Reset the list for the next partition

    logging.debug(prediction_df)
    return prediction_df

def merge_and_save_predictions(INTERMEDIATE_DIR, OUTPUT_DIR, PROMPTING_TYPE, PHASE, prompt_dict_keys):
    """
    Merges prediction DataFrames and saves them to an Excel file.

    Args:
        INTERMEDIATE_DIR (Path): Directory containing the JSONL files.
        OUTPUT_DIR (Path): Directory where the output Excel file will be saved.
        PROMPTING_TYPE (str): Type of prompting.
        PHASE (str): Phase of analysis ("prompt_opt", "final_pred", or "experimental").
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).

    Returns:
        Prediction results saved.
    """
    results_dict = {}
    
    if PROMPTING_TYPE in ["s_d", "cot_d"]:
        for key in prompt_dict_keys:
            results_dict[f"df_{key}"] = parse_result_jsonl(INTERMEDIATE_DIR, PROMPTING_TYPE, PHASE, key)
            logging.debug(f"Parsed predictions for {key}")
    
    elif PROMPTING_TYPE in ["s_m", "cot_m"]:
        typologies = ["construction", "currentuse", "storeys"]
    
        for key in typologies:
            results_dict[f"df_{key}"] = parse_result_jsonl(INTERMEDIATE_DIR, PROMPTING_TYPE, PHASE, key)
            logging.debug(f"Parsed predictions for {key}")
    
    merged_df = pd.DataFrame({"id": []})
    for i, value in results_dict.items():
        merged_df = pd.merge(merged_df, value, on='id', how='outer')

    out_path = OUTPUT_DIR / PHASE / f"{PROMPTING_TYPE}_google.xlsx"
    merged_df[~merged_df.id.duplicated()].to_excel(out_path, index=False)
    logging.info(f"Merged predictions saved to {out_path}")

In [3]:
# Configuration
INTERMEDIATE_DIR = Path("./temp/3_1")
OUTPUT_DIR = Path("./output/3_1")

# Create directories if they don't exist
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GOOGLE_RESPONSE_SCHEMAS_PATH = "./helper/google_response_schemas.json"
TYPOLOGIES_PREDICTION_HELPER_PATH = "./helper/typologies_prediction_helper.json"
ARD_GSV_PATH = "./output/2/ard"

BUCKET_URI = "gs://ci_pred"  # Insert your GCS Bucket URI
BUCKET_NAME = BUCKET_URI.split("//")[1]
PROJECT_ID = "colouring-indonesia"
LOCATION = "europe-west1"
MODEL_ID = "gemini-2.0-flash-001"

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./helper/google_credentials.json"
vertexai.init(project=PROJECT_ID, location=LOCATION)

In [4]:
PROMPTING_TYPE = "s_m"  # s_d: straight divided, cot_d: CoT divided, s_m: straight merged, cot_m: CoT merged

PREDICT_COUNT = 5  # GSV image to predict (2000 for prompt_opt phase, 30000 for final_pred phase, or otherwise for experimentation)
if PREDICT_COUNT == 2000:
    PHASE = "prompt_opt"
elif PREDICT_COUNT == 30000:
    PHASE = "final_pred"
else:
    PHASE = "experimental"

logger = setup_logging(INTERMEDIATE_DIR, PHASE)

In [5]:
NUM_PARTITION = 2  # Adjust later according to your needs and rate limit

In [6]:
response_schema_dict = get_response_schemas(GOOGLE_RESPONSE_SCHEMAS_PATH, PROMPTING_TYPE)
token, system_prompt_dict, prompt_dict, prompt_dict_keys, base_dataset = typologies_prediction_helpers(TYPOLOGIES_PREDICTION_HELPER_PATH, PROMPTING_TYPE)

2025-10-27 15:21:41,094 - INFO - Loaded response schemas from ./helper/google_response_schemas.json
2025-10-27 15:21:41,102 - DEBUG - Response schema for s_m prompting:
{'response_schema_merged': {'properties': {'construction': {'nullable': False,
                                                            'type': 'STRING'},
                                           'currentuse': {'nullable': False,
                                                          'type': 'STRING'},
                                           'reason_construction': {'nullable': False,
                                                                   'type': 'STRING'},
                                           'reason_currentuse': {'nullable': False,
                                                                 'type': 'STRING'},
                                           'reason_storeys': {'nullable': False,
                                                              'type': 'STRING'},
                 

In [ ]:
dataset = create_helper_df(ARD_GSV_PATH, PREDICT_COUNT, system_prompt_dict, prompt_dict, prompt_dict_keys, base_dataset)
tasks_dict = get_task_partitions(BUCKET_NAME, NUM_PARTITION, response_schema_dict, prompt_dict_keys, dataset, token)

In [ ]:
upload_requests_to_gcs(BUCKET_NAME,  INTERMEDIATE_DIR, NUM_PARTITION, PROMPTING_TYPE, PHASE, prompt_dict_keys)

In [ ]:
submit_prediction_request(BUCKET_URI, MODEL_ID, NUM_PARTITION, PROMPTING_TYPE, prompt_dict_keys)

In [ ]:
# Copy and paste BatchPredictionJob from the above printed objects after submitting each job
# Adjust TYPOLOGY accordingly (e.g., merged, currentuse, etc.)

metadata = {}
# Example: metadata[f"s_m_construction_partition_0"] = "projects/411353651824/locations/europe-west1/batchPredictionJobs/4310642002485051392"  # From above, look for: insert BatchPredictionJob to ((("projects/411353651824/locations/europe-west1/batchPredictionJobs/4310642002485051392")))
# metadata[f"{PROMPTING_TYPE}_TYPOLOGY_partition_1"] = "projects/411353651824/locations/europe-west1/batchPredictionJobs/"
# metadata[f"{PROMPTING_TYPE}_TYPOLOGY_partition_2"] = "projects/411353651824/locations/europe-west1/batchPredictionJobs/"

job_dict = {
    f"{PROMPTING_TYPE}_TYPOLOGY_partition_0": BatchPredictionJob(metadata[f"{PROMPTING_TYPE}_TYPOLOGY_partition_0"]).to_dict()["outputInfo"]["gcsOutputDirectory"],
    # f"{PROMPTING_TYPE}_TYPOLOGY_partition_1": BatchPredictionJob(metadata[f"{PROMPTING_TYPE}_TYPOLOGY_partition_1"]).to_dict()["outputInfo"]["gcsOutputDirectory"],
    # f"{PROMPTING_TYPE}_TYPOLOGY_partition_2": BatchPredictionJob(metadata[f"{PROMPTING_TYPE}_TYPOLOGY_partition_2"]).to_dict()["outputInfo"]["gcsOutputDirectory"],
}

In [ ]:
download_prediction_results(BUCKET_NAME, NUM_PARTITION, PROMPTING_TYPE, job_dict, prompt_dict_keys)

In [ ]:
merge_and_save_predictions(INTERMEDIATE_DIR, OUTPUT_DIR, PROMPTING_TYPE, PHASE, prompt_dict_keys)